# 末端笛卡尔直线插补:仿真与实机控制

让机械臂的 DH 末端帧沿一条笛卡尔直线从起点平滑运动到终点。

主要步骤:

1. 用当前标准 DH 表组装 RTB 的 `DHRobot`(`dfarm`)。
2. 调 `rtb.tools.trajectory.ctraj` 在起止位姿之间插出 N 段笛卡尔直线轨迹。
3. 对每个轨迹点用 `ikine_LM` 求逆解,前一点的解作为下一点的初值,以保证关节连续性。
4. 用 `dfarm.plot(..., movie=...)` 把动画保存成 gif。
5. 仿真无误后,打开 `execute_on_hardware = True` 把同一组关节序列下发到一体化关节。

注意:这里的坐标是 DH 末端帧原点(不一定等于夹爪 TCP)。

In [ ]:
# ---- 把存放 arm_robot.py 的目录加入 sys.path,使 import 可达 -----------------
import os, sys
_search_root = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isfile(os.path.join(_search_root, 'arm_robot.py')):
        if _search_root not in sys.path:
            sys.path.insert(0, _search_root)
        break
    _search_root = os.path.dirname(_search_root)
else:
    raise ImportError('未在最近 5 层父目录中定位到 arm_robot.py;请在 daran/ 或其子目录下打开本笔记本')
# ----------------------------------------------------------------------------
import time
import numpy as np
import roboticstoolbox as rtb
from roboticstoolbox import DHRobot, RevoluteDH
from spatialmath import SE3
import arm_robot as robot

np.set_printoptions(precision=4, suppress=True)

# ============================================================
# 1) 直线轨迹参数
# ============================================================
# DH 末端帧原点的起止笛卡尔点(mm)。这条 ~280 mm 的 3D 对角线沿 -x、-y、+z 方向走,
# 长度足够在实机运行时清晰看出直线轨迹。
path_start_mm  = [220,  120, 160]
path_end_mm    = [100, -120, 240]
waypoint_count = 40

# 是否仅约束位置(忽略起止位姿的 R/P/Y)
ignore_orientation   = True
start_orient_rpy_deg = [0, 0, 0]
end_orient_rpy_deg   = [0, 0, 0]
euler_convention     = 'zyx'

# 反解的初值(度)。建议尽量接近机械臂当前姿态,减少首点跳变。
ik_seed_deg = [0, 40, 40, 0, 0, 0]

# 关节模型角范围(度)
qmin_deg = [-160, -40, -160, -160, -180, -180]
qmax_deg = [ 160, 180,  160,  160,  180,  180]

# 动画输出
animation_filename = 'cartesian_line_anim.gif'
plot_box           = [0.05, 0.30, -0.20, 0.20, 0.10, 0.30]

# 实机控制
execute_on_hardware = False
can_bridge_port     = '/dev/ttyACM2'
serial_baud         = 115200

# 'tracking': 连续发送 mode=0 跟踪;适合小步长直线
# 'point'   : 每点用 mode=1 梯形轨迹到位;只在测试单点时用
runtime_mode      = 'tracking'
cmd_period_s      = 0.2
joint_speed_rpm   = 2.0
joint_accel_rpm_s = 10.0


# ============================================================
# 2) 模型 / 限位检查 / 位姿构造
# ============================================================
LINK_PARAMS = (
    dict(alpha=np.pi/2),
    dict(a=0.15),
    dict(a=0.15),
    dict(d=-0.05494, alpha=np.pi/2, offset=np.pi/2),
    dict(d=0.068,    alpha=-np.pi/2),
    dict(d=0.033),
)


def make_dfarm():
    links = [
        RevoluteDH(qlim=np.deg2rad([qmin_deg[i], qmax_deg[i]]), **kw)
        for i, kw in enumerate(LINK_PARAMS)
    ]
    return DHRobot(links, name='DFarm_StdDH')


def verify_within_limits(angles_deg):
    arr = np.asarray(angles_deg, dtype=float)
    lo  = np.asarray(qmin_deg,    dtype=float)
    hi  = np.asarray(qmax_deg,    dtype=float)
    bad = np.flatnonzero((arr < lo) | (arr > hi))
    if bad.size:
        msg = '; '.join(
            f'J{i+1}: {arr[i]:+.2f}° not in [{lo[i]:+.2f}, {hi[i]:+.2f}]'
            for i in bad
        )
        raise ValueError('关节模型角越界 -> ' + msg)


def target_pose_se3(xyz_mm, rpy_deg):
    pose = SE3.Trans(np.asarray(xyz_mm, dtype=float) * 1e-3)
    if not ignore_orientation:
        pose = pose * SE3.RPY(rpy_deg, unit='deg', order=euler_convention)
    return pose


# ============================================================
# 3) 生成轨迹
# ============================================================
dfarm        = make_dfarm()
T_start_pose = target_pose_se3(path_start_mm, start_orient_rpy_deg)
T_end_pose   = target_pose_se3(path_end_mm,   end_orient_rpy_deg)

cartesian_path = rtb.tools.trajectory.ctraj(T_start_pose, T_end_pose, waypoint_count)
dof_mask = [1, 1, 1, 0, 0, 0] if ignore_orientation else [1] * 6

joint_solutions = []
seed = np.deg2rad(np.asarray(ik_seed_deg, dtype=float))
for idx, T_step in enumerate(cartesian_path):
    ik = dfarm.ikine_LM(
        T_step, q0=seed, mask=dof_mask,
        ilimit=100, slimit=100, joint_limits=True,
    )
    if not ik.success:
        raise RuntimeError(
            f'第 {idx} 个轨迹点反解失败: {getattr(ik, "reason", "")}; '
            f'residual={getattr(ik, "residual", None)}'
        )
    seed = ik.q
    joint_solutions.append(ik.q)

joint_path     = np.asarray(joint_solutions)
joint_path_deg = np.rad2deg(joint_path)
verify_within_limits(joint_path_deg.min(axis=0))
verify_within_limits(joint_path_deg.max(axis=0))

# 末端 FK 与名义直线的位置残差
realized_xyz_mm = np.asarray([dfarm.fkine(q).t * 1000 for q in joint_path])
nominal_xyz_mm  = np.linspace(path_start_mm, path_end_mm, waypoint_count)
ik_residual_mm  = realized_xyz_mm - nominal_xyz_mm

print('轨迹点数            :', waypoint_count)
print('起点目标 mm         :', path_start_mm)
print('终点目标 mm         :', path_end_mm)
print('起点关节角 deg      :', np.round(joint_path_deg[0],  3).tolist())
print('终点关节角 deg      :', np.round(joint_path_deg[-1], 3).tolist())
print('每关节最小值 deg    :', np.round(joint_path_deg.min(axis=0), 3).tolist())
print('每关节最大值 deg    :', np.round(joint_path_deg.max(axis=0), 3).tolist())
print('IK 位置残差最大 mm  :', float(np.max(np.linalg.norm(ik_residual_mm, axis=1))))
print('首末 FK 坐标 mm     :')
print('  start:', np.round(realized_xyz_mm[0],  3).tolist())
print('  end  :', np.round(realized_xyz_mm[-1], 3).tolist())

# 表格化关节角(Jupyter 内显示;脚本/无显示环境会 fallback 到 numpy 输出)
try:
    import pandas as pd
    display(pd.DataFrame(joint_path_deg, columns=[f'J{i}' for i in range(1, 7)]).round(3))
except Exception:
    print(np.round(joint_path_deg, 3))

# 自定义动画循环:先把预期直线、起点、终点标注在 3D 坐标系上,
# 再逐帧捕获机械臂姿态。这样 gif 里就能直观看到末端是否沿红线运行。
from roboticstoolbox.backends.PyPlot import PyPlot

env = PyPlot()
env.launch(name='Cartesian Line', limits=plot_box)
env.add(dfarm)

expected_xyz_m = nominal_xyz_mm / 1000.0
env.ax.plot(
    expected_xyz_m[:, 0], expected_xyz_m[:, 1], expected_xyz_m[:, 2],
    color='red', linewidth=2.5, linestyle='-', alpha=0.9,
    label='expected line',
)
env.ax.scatter(
    [expected_xyz_m[0, 0]], [expected_xyz_m[0, 1]], [expected_xyz_m[0, 2]],
    color='limegreen', s=80, marker='o', label='start',
)
env.ax.scatter(
    [expected_xyz_m[-1, 0]], [expected_xyz_m[-1, 1]], [expected_xyz_m[-1, 2]],
    color='magenta', s=80, marker='X', label='end',
)
env.ax.legend(loc='upper left', fontsize=8)

frames = []
for q in joint_path:
    dfarm.q = q
    env.step(0.05)
    frames.append(env.getframe())

# loop=0 表示 gif 无限循环;duration 单位是 ms
frames[0].save(
    animation_filename,
    save_all=True,
    append_images=frames[1:],
    optimize=False,
    duration=80,
    loop=0,
)
print('动画文件已保存(含预期直线标注):', animation_filename)

## 实机执行

确认上方动画与关节角范围合理后,把 `execute_on_hardware` 改成 True,再运行下一段。

In [ ]:
def query_joint_state(arm):
    raw = arm.read_joints()
    if raw is False:
        return False, False
    return raw, arm.servo_to_model(raw)


if not execute_on_hardware:
    print('execute_on_hardware = False,未发送实机命令。'
          '确认动画和关节角后改成 True 再运行。')
else:
    verify_within_limits(joint_path_deg.min(axis=0))
    verify_within_limits(joint_path_deg.max(axis=0))

    arm = robot.arm_robot(
        L_p             = 0,
        L_p_mass_center = 0,
        MAX_list_temp   = qmax_deg,
        MIN_list_temp   = qmin_deg,
        G_p             = 0,
        com             = can_bridge_port,
        uart_baudrate   = serial_baud,
    )

    raw_before, model_before = query_joint_state(arm)
    print('运动前 model =', model_before)
    if model_before is False:
        raise RuntimeError('读取当前关节角失败,中止运动。')

    # 先把机械臂运动到轨迹起点
    print('先运动到轨迹起点:', np.round(joint_path_deg[0], 3).tolist())
    if arm.set_arm_joints(angle_list=joint_path_deg[0].tolist(),
                          speed=joint_speed_rpm) is False:
        raise RuntimeError('运动到起点失败。')
    arm.pose_done()
    time.sleep(0.5)

    if runtime_mode == 'tracking':
        # mode=0 时 param 是输入滤波带宽,需 < 300,设为发送频率的一半
        filter_bw = min(250, max(1, 1.0 / (2.0 * cmd_period_s)))
        print(f'tracking 模式: cmd_period={cmd_period_s} s, filter_bw={filter_bw}')
        for q_deg_step in joint_path_deg:
            servo_cmd = arm.model_to_servo(q_deg_step.tolist())
            arm.set_angles(arm.ID_list, servo_cmd,
                           speed=20, param=filter_bw, mode=0)
            time.sleep(cmd_period_s)
    elif runtime_mode == 'point':
        print('point 模式 (逐点梯形)')
        for q_deg_step in joint_path_deg:
            servo_cmd = arm.model_to_servo(q_deg_step.tolist())
            arm.set_angles(arm.ID_list, servo_cmd,
                           speed=joint_speed_rpm, param=joint_accel_rpm_s, mode=1)
            time.sleep(cmd_period_s)
    else:
        raise ValueError("runtime_mode 只能取 'tracking' 或 'point'")

    time.sleep(0.5)
    raw_after, model_after = query_joint_state(arm)
    print('运动后 model =', model_after)
    if model_after is not False:
        T_done = dfarm.fkine(np.deg2rad(model_after))
        print('回读模型角对应的 FK 末端 mm =',
              np.round(T_done.t * 1000, 3).tolist())

## 可选:回到关节零位

把 `request_home = True` 后运行下面 cell,机械臂会运动回 [0, 0, 0, 0, 0, 0]。

In [ ]:
request_home  = False
home_pose_deg = [0, 0, 0, 0, 0, 0]

if not request_home:
    print('request_home = False,未发送回零命令。')
else:
    target = np.asarray(home_pose_deg, dtype=float)
    verify_within_limits(target)
    arm = robot.arm_robot(
        L_p             = 0,
        L_p_mass_center = 0,
        MAX_list_temp   = qmax_deg,
        MIN_list_temp   = qmin_deg,
        G_p             = 0,
        com             = can_bridge_port,
        uart_baudrate   = serial_baud,
    )
    print('运动前 model =', query_joint_state(arm)[1])
    sent_ok = arm.set_arm_joints(angle_list=target.tolist(), speed=joint_speed_rpm)
    print('指令下发结果 =', sent_ok)
    arm.pose_done()
    time.sleep(0.5)
    print('运动后 model =', query_joint_state(arm)[1])